# ☁️ Boto3 — AWS Automation
## Python Ecosystem Tutorial Series — Module 13 of 18

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

| | |
|---|---|
| **Library** | ☁️ Boto3 |
| **Domain** | AWS Automation |
| **Dataset** | S3, EC2, SageMaker |
| **Module** | 13 of 18 |

**What you will learn:**

1. What Boto3 is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install boto3
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `boto3.client("s3")` | S3 client |
| `s3.upload_file()` | Upload to S3 |
| `s3.download_file()` | Download from S3 |
| `ec2.create_instances()` | Launch EC2 |
| `instance.stop()` | Stop (save money!) |

# 13. ☁️ Boto3 — AWS Automation
> **Python + Boto3 = AWS Automation**

Boto3 is the official AWS SDK for Python. Automate cloud infrastructure:
S3 buckets, EC2 instances, Lambda functions, DynamoDB, SageMaker.

**Key concepts:** boto3.client, boto3.resource, S3 upload/download, EC2 lifecycle

In [ ]:
# ── Boto3 patterns — key patterns (requires AWS credentials to run live) ──────
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

print("Boto3 AWS Automation Patterns")
print("="*50)
print("(Requires: pip install boto3 + AWS credentials in ~/.aws/credentials)")
print()

boto3_patterns = """
# ── Install ────────────────────────────────────────────────────────────────────
pip install boto3

# ── Configure credentials (one-time setup) ─────────────────────────────────────
aws configure
# Enter: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, region, output format

import boto3

# ══════════════════════════════════════════════════════════════════════════════
# PATTERN 1: S3 — Upload / Download files
# ══════════════════════════════════════════════════════════════════════════════
s3 = boto3.client("s3")

# Create a bucket
s3.create_bucket(Bucket="my-data-bucket",
                  CreateBucketConfiguration={"LocationConstraint": "us-west-2"})

# Upload a file
s3.upload_file("local_data.csv", "my-data-bucket", "processed/data.csv")

# Download a file
s3.download_file("my-data-bucket", "processed/data.csv", "data_downloaded.csv")

# List objects
response = s3.list_objects_v2(Bucket="my-data-bucket", Prefix="processed/")
for obj in response["Contents"]:
    print(f"{obj['Key']} — {obj['Size']/1024:.1f} KB")

# ══════════════════════════════════════════════════════════════════════════════
# PATTERN 2: EC2 — Start / Stop / Monitor instances
# ══════════════════════════════════════════════════════════════════════════════
ec2 = boto3.resource("ec2")

# Launch an instance
instances = ec2.create_instances(
    ImageId="ami-0c55b159cbfafe1f0",  # Amazon Linux 2
    MinCount=1,
    MaxCount=1,
    InstanceType="t3.medium",          # 2 vCPU, 4 GB RAM
    KeyName="my-key-pair",
    TagSpecifications=[{"ResourceType":"instance",
                         "Tags":[{"Key":"Name","Value":"ML-Training-Server"}]}]
)
instance = instances[0]
instance.wait_until_running()
print(f"Instance {instance.id} running at {instance.public_ip_address}")

# Stop when done (saves money!)
instance.stop()

# ══════════════════════════════════════════════════════════════════════════════
# PATTERN 3: Automate ML pipeline on SageMaker
# ══════════════════════════════════════════════════════════════════════════════
sagemaker = boto3.client("sagemaker")

# Start a training job
sagemaker.create_training_job(
    TrainingJobName="toxicity-prediction-v1",
    AlgorithmSpecification={"TrainingImage":"683313688378.dkr.ecr.us-east-1.amazonaws.com/xgboost:1.5-1",
                              "TrainingInputMode":"File"},
    RoleArn="arn:aws:iam::123456789:role/SageMakerRole",
    InputDataConfig=[{"ChannelName":"training",
                       "DataSource":{"S3DataSource":{"S3Uri":"s3://my-bucket/train/"}}}],
    OutputDataConfig={"S3OutputPath":"s3://my-bucket/output/"},
    ResourceConfig={"InstanceType":"ml.m5.xlarge","InstanceCount":1,"VolumeSizeInGB":30}
)

# ══════════════════════════════════════════════════════════════════════════════
# PATTERN 4: DynamoDB — NoSQL database operations
# ══════════════════════════════════════════════════════════════════════════════
dynamodb = boto3.resource("dynamodb")
table    = dynamodb.Table("ChemicalRegistry")

table.put_item(Item={"dtxsid":"DTXSID7020182","name":"Bisphenol A","mw":228.3,"logp":3.32})
response = table.get_item(Key={"dtxsid":"DTXSID7020182"})
print(response["Item"])
"""
print(boto3_patterns)

# ── Architecture diagram ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14); ax.set_ylim(0, 7); ax.axis("off")
ax.set_facecolor("#F0F4F8")

# Draw AWS services
services = [
    (1.5, 5.5, "S3\nFile Storage", "#FF9900", "📦"),
    (5.5, 5.5, "EC2\nCompute", "#FF9900", "💻"),
    (9.5, 5.5, "SageMaker\nML Training", "#FF9900", "🧠"),
    (1.5, 2.5, "DynamoDB\nDatabase", "#FF9900", "🗄️"),
    (5.5, 2.5, "Lambda\nServerless", "#FF9900", "λ"),
    (9.5, 2.5, "CloudWatch\nMonitoring", "#FF9900", "📊"),
]

for x, y, label, col, emoji in services:
    rect = mpatches.FancyBboxPatch((x-1.2, y-0.7), 2.4, 1.4,
                                    boxstyle="round,pad=0.15",
                                    facecolor=col, edgecolor="white",
                                    alpha=0.85, lw=2)
    ax.add_patch(rect)
    ax.text(x, y+0.2, emoji, ha="center", va="center", fontsize=18)
    ax.text(x, y-0.3, label, ha="center", va="center",
             fontsize=8, fontweight="bold", color="white")

# Python boto3 box
py_rect = mpatches.FancyBboxPatch((5.5, 1.2) , 3, 0.8,
                                    boxstyle="round,pad=0.1",
                                    facecolor="#3776AB", edgecolor="white", lw=2)
ax.add_patch(py_rect)
ax.text(7, 1.6, "🐍  Python  boto3", ha="center", va="center",
         fontsize=11, fontweight="bold", color="white")

# Arrows
for x, y in [(1.5,5.5),(5.5,5.5),(9.5,5.5),(1.5,2.5),(5.5,2.5),(9.5,2.5)]:
    ax.annotate("", xy=(x, y-0.7), xytext=(7, 2.0),
                 arrowprops=dict(arrowstyle="-", color="#3776AB", lw=1.5, alpha=0.4))

ax.text(7, 6.5, "AWS Cloud Services — Controlled via Python boto3",
         ha="center", fontsize=12, fontweight="bold", color="#1F2937")

plt.tight_layout()
plt.savefig("boto3_aws.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: Boto3

### AWS SDK Request Flow
Every boto3 call: Python -> sign request with credentials -> HTTPS to AWS API endpoint -> parse JSON response -> return Python dict. All of this is handled automatically by boto3.

### client vs resource
Client is the low-level interface that mirrors the AWS API exactly (returns plain Python dicts). Resource is the object-oriented interface that's more Pythonic (returns typed objects with methods). Both do the same thing — choose whichever feels more natural.

### Credential Chain (in order)
1. Explicit in code (bad for production — hardcoded secrets)
2. Environment variables AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY
3. ~/.aws/credentials file (from `aws configure`)
4. IAM Instance Role (best practice on EC2/Lambda — no credentials stored anywhere)

### S3 Key Patterns for Data Science
```python
# Organise by project/date/type
"projects/toxicology/2024-01-15/raw/chemicals.csv"
"projects/toxicology/2024-01-15/models/rf_v1.pkl"
"projects/toxicology/2024-01-15/results/predictions.csv"
```

### EC2 Cost Management
```python
instance.stop()     # stopped: pay only for storage (~$0.10/month)
instance.start()    # running: pay by the second
instance.terminate() # terminated: deleted, pay nothing
```
A forgotten GPU instance (p3.2xlarge) costs $3/hour = $2,190/month.


## ✅ Key Takeaways — ☁️ Boto3

1. Never hardcode AWS credentials — use IAM roles in production
2. Always call instance.stop() when EC2 work is done — forgotten instances are expensive
3. S3 is the universal storage layer — everything else reads/writes from it
4. boto3.resource() is more Pythonic; boto3.client() gives more control

---
*Next: Continue to Module 14 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [himanshugoel.github.io](https://himanshugoel.github.io)*